# 05. 모델링

`04_eda`까지 생성한 파생변수 포함 데이터셋을 사용해 실거래가 예측 모델을 학습한다.

진행 순서:

1. 최종 파생변수 데이터셋 로드
2. 모델 학습용 컬럼 정리 및 결측 처리
3. 거래일 기준 시간 분할
4. 기준 모델 학습 및 성능 비교
5. 변수 중요도 저장

In [78]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(PROJECT_ROOT / '.cache'))
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count() or 1))
(PROJECT_ROOT / '.matplotlib').mkdir(exist_ok=True)
(PROJECT_ROOT / '.cache').mkdir(exist_ok=True)

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

DATA_PATH = PROJECT_ROOT / 'data/processed/seoul_apt_trade_2025_features.csv'
MODELING_DATA_PATH = PROJECT_ROOT / 'data/processed/modeling_dataset.csv'
SCORE_PATH = PROJECT_ROOT / 'reports/model_scores.csv'
PRICE_BAND_MAE_PATH = PROJECT_ROOT / 'reports/model_price_band_mae.csv'
PREDICTION_PATH = PROJECT_ROOT / 'reports/model_test_predictions.csv'
FIGURE_DIR = PROJECT_ROOT / 'reports/figures'
IMPORTANCE_PATH = FIGURE_DIR / 'model_feature_importance.png'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 100)

## 1. 최종 파생변수 데이터 로드

모델링은 `03_feature_engineering`에서 만든 최종 데이터셋을 기준으로 한다.

In [79]:
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')
print(f'rows: {len(df):,}')
print(f'columns: {len(df.columns):,}')
df.head()

rows: 77,359
columns: 31


,sigungu,apartment_name,area_m2,contract_ym,contract_day,price_10k_krw,floor,built_year,road_name,sido,gu,law_dong,contract_year,contract_month,contract_date,age,price_per_m2_10k_krw,full_road_address,geocode_status,matched_address,latitude,longitude,distance_to_cbd_km,distance_to_ybd_km,distance_to_gbd_km,nearest_business_district_distance_km,nearest_business_district,nearest_subway_distance_km,hospital_count_within_1km,nearest_hospital_distance_km,large_mart_count_within_1km
0,서울특별시 성동구 하왕십리동,왕십리KCC스위첸,64.236,202512,31,137500,8,2016,무학봉길 35,서울특별시,성동구,하왕십리동,2025,12,2025-12-31,9,2140.544243,서울특별시 성동구 무학봉길 35,ok,서울특별시 성동구 무학봉길 35 왕십리KCC스위첸,37.560960,127.026967,4.359801,10.131561,7.012161,4.359801,CBD,0.432088,1.0,0.470749,0.0
1,서울특별시 종로구 행촌동,대성아파트,94.940,202512,31,69500,3,1971,사직로 21,서울특별시,종로구,행촌동,2025,12,2025-12-31,54,732.041289,서울특별시 종로구 사직로 21,ok,서울특별시 종로구 사직로 21 대성아파트,37.572812,126.962566,1.530684,6.379895,10.112848,1.530684,CBD,0.467477,3.0,0.521502,0.0
2,서울특별시 중구 충무로4가,남산센트럴자이,82.413,202512,31,128000,18,2009,퇴계로 235,서울특별시,중구,충무로4가,2025,12,2025-12-31,16,1553.153022,서울특별시 중구 퇴계로 235,ok,서울특별시 중구 퇴계로 235 남산 센트럴 자이,37.562514,126.997824,1.802663,7.910221,7.649539,1.802663,CBD,0.352045,1.0,0.866817,0.0
3,서울특별시 성동구 마장동,현대,84.910,202512,31,118000,12,1998,살곶이길 50,서울특별시,성동구,마장동,2025,12,2025-12-31,27,1389.706748,서울특별시 성동구 살곶이길 50,ok,서울특별시 성동구 살곶이길 50 청계현대아파트,37.569933,127.042494,5.697082,11.785014,8.116617,5.697082,CBD,0.351577,0.0,1.151700,0.0
4,서울특별시 중구 충무로4가,남산센트럴자이,80.473,202512,31,125000,8,2009,퇴계로 235,서울특별시,중구,충무로4가,2025,12,2025-12-31,16,1553.316019,서울특별시 중구 퇴계로 235,ok,서울특별시 중구 퇴계로 235 남산 센트럴 자이,37.562514,126.997824,1.802663,7.910221,7.649539,1.802663,CBD,0.352045,1.0,0.866817,0.0


## 2. 모델 학습용 데이터셋 구성

좌표 생성, 주소 확인, 사후 검증용 컬럼은 모델 입력에서 제외한다. 타깃은 실거래가 총액인 `price_10k_krw`를 사용한다.

`price_per_m2_10k_krw`는 타깃인 거래가를 면적으로 나눈 값이라 누수 가능성이 있으므로 입력 변수에서 제외한다.

In [ ]:
target_col = 'price_10k_krw'

numeric_features = [
    'area_m2',
    'floor',
    'built_year',
    'age',
    'contract_year',
    'contract_month',
    'contract_day',
    'distance_to_cbd_km',
    'distance_to_ybd_km',
    'distance_to_gbd_km',
    # 'nearest_business_district_distance_km',
    'nearest_subway_distance_km',
    # 'hospital_count_within_1km',
    'nearest_hospital_distance_km',
    'large_mart_count_within_1km',
]

categorical_features = [
    'gu',
    'law_dong',
    # 'apartment_name',
    # 'nearest_business_district',
]

required_columns = [target_col, 'contract_date', *numeric_features, *categorical_features]
modeling_df = df[required_columns].copy()
modeling_df['contract_date'] = pd.to_datetime(modeling_df['contract_date'])

missing_before = modeling_df.isna().sum().sort_values(ascending=False)
missing_before[missing_before > 0]

nearest_business_district_distance_km    8
nearest_subway_distance_km               8
nearest_hospital_distance_km             8
large_mart_count_within_1km              8
nearest_business_district                8
dtype: int64

In [81]:
rows_before = len(modeling_df)
modeling_df = modeling_df.dropna(subset=[target_col, *numeric_features, *categorical_features]).copy()
rows_after = len(modeling_df)

modeling_df.to_csv(MODELING_DATA_PATH, index=False, encoding='utf-8-sig')

print(f'모델링 데이터 저장: {MODELING_DATA_PATH}')
print(f'제거된 행: {rows_before - rows_after:,}')
print(f'최종 행 수: {rows_after:,}')
modeling_df.head()

모델링 데이터 저장: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/data/processed/modeling_dataset.csv
제거된 행: 8
최종 행 수: 77,351


,price_10k_krw,contract_date,area_m2,floor,built_year,age,contract_year,contract_month,contract_day,nearest_business_district_distance_km,nearest_subway_distance_km,nearest_hospital_distance_km,large_mart_count_within_1km,gu,law_dong,nearest_business_district
0,137500,2025-12-31,64.236,8,2016,9,2025,12,31,4.359801,0.432088,0.470749,0.0,성동구,하왕십리동,CBD
1,69500,2025-12-31,94.940,3,1971,54,2025,12,31,1.530684,0.467477,0.521502,0.0,종로구,행촌동,CBD
2,128000,2025-12-31,82.413,18,2009,16,2025,12,31,1.802663,0.352045,0.866817,0.0,중구,충무로4가,CBD
3,118000,2025-12-31,84.910,12,1998,27,2025,12,31,5.697082,0.351577,1.151700,0.0,성동구,마장동,CBD
4,125000,2025-12-31,80.473,8,2009,16,2025,12,31,1.802663,0.352045,0.866817,0.0,중구,충무로4가,CBD


## 3. 학습/테스트 데이터 분할

실제 예측 상황과 비슷하게 거래일 기준 시간 분할을 사용한다. 2025년 1월부터 10월까지는 학습, 2025년 11월부터 12월까지는 테스트로 둔다.

In [82]:
split_date = pd.Timestamp('2025-11-01')
train_df = modeling_df[modeling_df['contract_date'] < split_date].copy()
test_df = modeling_df[modeling_df['contract_date'] >= split_date].copy()

X_train = train_df[numeric_features + categorical_features]
y_train = train_df[target_col]
X_test = test_df[numeric_features + categorical_features]
y_test = test_df[target_col]

split_summary = pd.DataFrame({
    'dataset': ['train', 'test'],
    'start_date': [train_df['contract_date'].min(), test_df['contract_date'].min()],
    'end_date': [train_df['contract_date'].max(), test_df['contract_date'].max()],
    'rows': [len(train_df), len(test_df)],
    'target_mean': [y_train.mean(), y_test.mean()],
    'target_median': [y_train.median(), y_test.median()],
})
split_summary

,dataset,start_date,end_date,rows,target_mean,target_median
0,train,2025-01-01,2025-10-31,69197,127982.512725,105000.0
1,test,2025-11-01,2025-12-31,8154,117914.702968,93000.0


## 4. 모델 파이프라인 정의

설치된 기본 라이브러리만 사용하기 위해 `scikit-learn` 모델로 기준선을 만든다.

- `DummyRegressor`: 평균값 예측 기준선
- `LinearRegression`: 선형회귀
- `RandomForestRegressor`: 랜덤 포레스트
- `ExtraTreesRegressor`: 엑스트라 트리
- `GradientBoostingRegressor`: 그래디언트 부스팅
- `HistGradientBoostingRegressor`: 히스토그램 기반 그래디언트 부스팅

In [83]:
onehot_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), categorical_features),
    ],
    remainder='drop',
)

ordinal_preprocessor = ColumnTransformer(
    transformers=[
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), categorical_features),
    ],
    remainder='drop',
)

models = {
    'dummy_mean': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', DummyRegressor(strategy='mean')),
    ]),
    'linear_regression': Pipeline([
        ('preprocess', onehot_preprocessor),
        ('model', LinearRegression()),
    ]),
    'random_forest': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', RandomForestRegressor(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'extra_trees': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', ExtraTreesRegressor(
            n_estimators=120,
            max_depth=18,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )),
    ]),
    'gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.06,
            max_depth=4,
            min_samples_leaf=3,
            random_state=42,
        )),
    ]),
    'hist_gradient_boosting': Pipeline([
        ('preprocess', ordinal_preprocessor),
        ('model', HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.1,
            random_state=42,
        )),
    ]),
}

## 5. 모델 학습 및 평가

평가 지표는 회귀 문제에서 자주 쓰는 `MAE`, `RMSE`, `R2`를 사용한다. 금액 단위는 원본 타깃과 같은 만 원이다.

In [84]:
def evaluate_regression(y_true, y_pred):
    return {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred),
    }


def fit_and_evaluate_models(X_train, y_train, X_test, y_test, split_strategy):
    scores = []
    fitted = {}

    for model_name, model in models.items():
        print(f'training: {split_strategy} / {model_name}')
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        fitted[model_name] = fitted_model

        train_pred = fitted_model.predict(X_train)
        test_pred = fitted_model.predict(X_test)

        for dataset_name, y_true, y_pred in [
            ('train', y_train, train_pred),
            ('test', y_test, test_pred),
        ]:
            scores.append({
                'split_strategy': split_strategy,
                'model': model_name,
                'dataset': dataset_name,
                **evaluate_regression(y_true, y_pred),
            })

    return pd.DataFrame(scores), fitted


def make_test_prediction_df(fitted, X_test, y_test, split_strategy):
    rows = []
    for model_name, model in fitted.items():
        pred = model.predict(X_test)
        rows.append(pd.DataFrame({
            'split_strategy': split_strategy,
            'model': model_name,
            'actual_price_10k_krw': y_test.to_numpy(),
            'predicted_price_10k_krw': pred,
            'absolute_error_10k_krw': np.abs(y_test.to_numpy() - pred),
        }))
    return pd.concat(rows, ignore_index=True)

score_df, fitted_models = fit_and_evaluate_models(
    X_train,
    y_train,
    X_test,
    y_test,
    split_strategy='time_2025_01_10_train_11_12_test',
)

test_prediction_df = make_test_prediction_df(
    fitted_models,
    X_test,
    y_test,
    split_strategy='time_2025_01_10_train_11_12_test',
)

score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 저장: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

training: time_2025_01_10_train_11_12_test / dummy_mean
training: time_2025_01_10_train_11_12_test / linear_regression
training: time_2025_01_10_train_11_12_test / random_forest
training: time_2025_01_10_train_11_12_test / extra_trees
training: time_2025_01_10_train_11_12_test / gradient_boosting
training: time_2025_01_10_train_11_12_test / hist_gradient_boosting
성능표 저장: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/reports/model_scores.csv


,split_strategy,model,dataset,mae,rmse,r2
7,time_2025_01_10_train_11_12_test,extra_trees,test,11469.523563,20059.807583,0.951790
5,time_2025_01_10_train_11_12_test,random_forest,test,11083.918145,20612.263524,0.949098
11,time_2025_01_10_train_11_12_test,hist_gradient_boosting,test,18047.971987,26696.106363,0.914615
9,time_2025_01_10_train_11_12_test,gradient_boosting,test,24830.799491,37758.477549,0.829189
3,time_2025_01_10_train_11_12_test,linear_regression,test,30204.269225,48959.501652,0.712815
1,time_2025_01_10_train_11_12_test,dummy_mean,test,63842.538337,91913.109603,-0.012144
6,time_2025_01_10_train_11_12_test,extra_trees,train,5901.169317,12305.601997,0.983284
4,time_2025_01_10_train_11_12_test,random_forest,train,5298.637385,12567.740170,0.982564
10,time_2025_01_10_train_11_12_test,hist_gradient_boosting,train,13156.172412,21434.174919,0.949284
8,time_2025_01_10_train_11_12_test,gradient_boosting,train,20906.596493,32231.579769,0.885318


## 7. 랜덤 80:20 분할 성능 비교

전체 데이터를 섞은 뒤 80%는 훈련셋, 20%는 테스트셋으로 사용한다. 시간 분할보다 일반적인 교차 검증 상황에 가까운 비교용 결과다.

In [85]:
X = modeling_df[numeric_features + categorical_features]
y = modeling_df[target_col]

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

random_split_summary = pd.DataFrame({
    'dataset': ['train_random', 'test_random'],
    'rows': [len(X_train_random), len(X_test_random)],
    'target_mean': [y_train_random.mean(), y_test_random.mean()],
    'target_median': [y_train_random.median(), y_test_random.median()],
})
random_split_summary

,dataset,rows,target_mean,target_median
0,train_random,61880,126900.184615,104500.0
1,test_random,15471,127005.300045,104000.0


In [86]:
random_score_df, random_fitted_models = fit_and_evaluate_models(
    X_train_random,
    y_train_random,
    X_test_random,
    y_test_random,
    split_strategy='random_80_20',
)

random_prediction_df = make_test_prediction_df(
    random_fitted_models,
    X_test_random,
    y_test_random,
    split_strategy='random_80_20',
)
test_prediction_df = pd.concat([test_prediction_df, random_prediction_df], ignore_index=True)

score_df = pd.concat([score_df, random_score_df], ignore_index=True)
score_df.to_csv(SCORE_PATH, index=False, encoding='utf-8-sig')
print(f'성능표 업데이트: {SCORE_PATH}')
score_df.sort_values(['split_strategy', 'dataset', 'rmse'])

training: random_80_20 / dummy_mean
training: random_80_20 / linear_regression
training: random_80_20 / random_forest
training: random_80_20 / extra_trees
training: random_80_20 / gradient_boosting
training: random_80_20 / hist_gradient_boosting
성능표 업데이트: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/reports/model_scores.csv


,split_strategy,model,dataset,mae,rmse,r2
19,random_80_20,extra_trees,test,8390.457656,18029.294783,0.964179
17,random_80_20,random_forest,test,8205.354502,18314.340879,0.963037
23,random_80_20,hist_gradient_boosting,test,14568.539653,24188.098912,0.935526
21,random_80_20,gradient_boosting,test,21736.443588,34617.944818,0.867936
15,random_80_20,linear_regression,test,26581.475133,44952.026667,0.777320
13,random_80_20,dummy_mean,test,62637.468529,95259.636624,-0.000001
18,random_80_20,extra_trees,train,6101.380907,12670.107590,0.982109
16,random_80_20,random_forest,train,5498.575912,12830.549846,0.981653
22,random_80_20,hist_gradient_boosting,train,13479.892717,21797.050329,0.947051
20,random_80_20,gradient_boosting,train,20717.700797,31961.119025,0.886156


## 8. 분할 방식별 테스트 성능 비교

In [87]:
test_scores = score_df[score_df['dataset'].eq('test')].sort_values(['split_strategy', 'rmse']).reset_index(drop=True)
test_scores

,split_strategy,model,dataset,mae,rmse,r2
0,random_80_20,extra_trees,test,8390.457656,18029.294783,0.964179
1,random_80_20,random_forest,test,8205.354502,18314.340879,0.963037
2,random_80_20,hist_gradient_boosting,test,14568.539653,24188.098912,0.935526
3,random_80_20,gradient_boosting,test,21736.443588,34617.944818,0.867936
4,random_80_20,linear_regression,test,26581.475133,44952.026667,0.777320
5,random_80_20,dummy_mean,test,62637.468529,95259.636624,-0.000001
6,time_2025_01_10_train_11_12_test,extra_trees,test,11469.523563,20059.807583,0.951790
7,time_2025_01_10_train_11_12_test,random_forest,test,11083.918145,20612.263524,0.949098
8,time_2025_01_10_train_11_12_test,hist_gradient_boosting,test,18047.971987,26696.106363,0.914615
9,time_2025_01_10_train_11_12_test,gradient_boosting,test,24830.799491,37758.477549,0.829189


In [88]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_df = test_scores.sort_values(['split_strategy', 'rmse'], ascending=[True, False]).copy()
plot_df['label'] = plot_df['split_strategy'] + ' / ' + plot_df['model']
ax.barh(plot_df['label'], plot_df['rmse'], color='#4C78A8')
ax.set_xlabel('RMSE (10k KRW)')
ax.set_ylabel('split / model')
ax.set_title('Test RMSE by Split Strategy')
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
plt.show()
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_58113/990993299.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. 가격대별 MAE 확인

전체 MAE만 보면 저가·고가 구간 중 어느 구간에서 오차가 큰지 알기 어렵다. 테스트셋의 실제 거래가를 가격 구간으로 나누고, 구간별 평균 절대 오차를 확인한다.

In [89]:
price_bins = [0, 50_000, 100_000, 150_000, 200_000, 300_000, np.inf]
price_labels = [
    '<=5eok',
    '5-10eok',
    '10-15eok',
    '15-20eok',
    '20-30eok',
    '>30eok',
]

test_prediction_df['price_band'] = pd.cut(
    test_prediction_df['actual_price_10k_krw'],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

price_band_mae = (
    test_prediction_df
    .groupby(['split_strategy', 'model', 'price_band'], observed=True)
    .agg(
        rows=('absolute_error_10k_krw', 'size'),
        actual_mean=('actual_price_10k_krw', 'mean'),
        pred_mean=('predicted_price_10k_krw', 'mean'),
        mae=('absolute_error_10k_krw', 'mean'),
    )
    .reset_index()
    .sort_values(['split_strategy', 'model', 'price_band'])
)

price_band_mae.to_csv(PRICE_BAND_MAE_PATH, index=False, encoding='utf-8-sig')
print(f'가격대별 MAE 저장: {PRICE_BAND_MAE_PATH}')
price_band_mae

가격대별 MAE 저장: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/reports/model_price_band_mae.csv


,split_strategy,model,price_band,rows,actual_mean,pred_mean,mae
0,random_80_20,dummy_mean,<=5eok,1731,35170.814558,126900.184615,91729.370057
1,random_80_20,dummy_mean,5-10eok,5739,75624.405297,126900.184615,51275.779318
2,random_80_20,dummy_mean,10-15eok,4003,124045.843867,126900.184615,12466.059113
3,random_80_20,dummy_mean,15-20eok,1970,172578.586802,126900.184615,45678.402187
4,random_80_20,dummy_mean,20-30eok,1291,245474.449264,126900.184615,118574.264649
...,...,...,...,...,...,...,...
67,time_2025_01_10_train_11_12_test,random_forest,5-10eok,3420,74387.986842,74812.409113,6162.608086
68,time_2025_01_10_train_11_12_test,random_forest,10-15eok,1880,123836.756383,118867.283128,12032.985598
69,time_2025_01_10_train_11_12_test,random_forest,15-20eok,851,173651.964747,166435.075343,18082.080296
70,time_2025_01_10_train_11_12_test,random_forest,20-30eok,536,243921.007463,230660.500026,24131.624903


In [90]:
best_models = test_scores.groupby('split_strategy').first().reset_index()[['split_strategy', 'model']]
best_price_band_mae = price_band_mae.merge(best_models, on=['split_strategy', 'model'])
best_price_band_mae

,split_strategy,model,price_band,rows,actual_mean,pred_mean,mae
0,random_80_20,extra_trees,<=5eok,1731,35170.814558,38390.571628,4095.189295
1,random_80_20,extra_trees,5-10eok,5739,75624.405297,78184.682236,5119.976568
2,random_80_20,extra_trees,10-15eok,4003,124045.843867,125254.860157,7230.495625
3,random_80_20,extra_trees,15-20eok,1970,172578.586802,171379.833648,10312.492463
4,random_80_20,extra_trees,20-30eok,1291,245474.449264,241531.296272,15897.373467
5,random_80_20,extra_trees,>30eok,737,429534.616011,415777.061418,31958.831639
6,time_2025_01_10_train_11_12_test,extra_trees,<=5eok,1079,33663.158480,36113.405831,4330.416815
7,time_2025_01_10_train_11_12_test,extra_trees,5-10eok,3420,74387.986842,74959.041338,6048.272814
8,time_2025_01_10_train_11_12_test,extra_trees,10-15eok,1880,123836.756383,118113.664380,12270.344731
9,time_2025_01_10_train_11_12_test,extra_trees,15-20eok,851,173651.964747,163473.710399,19185.893174


In [91]:
fig, axes = plt.subplots(1, len(best_models), figsize=(14, 4.5), sharey=True)
if len(best_models) == 1:
    axes = [axes]

for ax, (_, row) in zip(axes, best_models.iterrows()):
    plot_df = best_price_band_mae[
        best_price_band_mae['split_strategy'].eq(row['split_strategy'])
        & best_price_band_mae['model'].eq(row['model'])
    ]
    ax.bar(plot_df['price_band'].astype(str), plot_df['mae'], color='#4C78A8')
    ax.set_title(f"{row['split_strategy']} / {row['model']}")
    ax.set_xlabel('Actual price band')
    ax.tick_params(axis='x', rotation=35)
    ax.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('MAE (10k KRW)')
fig.tight_layout()
plt.show()
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_58113/2934026892.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. log-price 실험

`RandomForestRegressor`와 `ExtraTreesRegressor`를 기준으로 원가격을 직접 예측한 경우와 `log1p(price)`를 예측한 뒤 다시 가격 단위로 되돌린 경우를 비교한다. 이 실험은 최종 성능표에 저장하지 않고, log 변환이 실제로 도움이 되는지 확인하는 용도로만 사용한다.

In [92]:
def fit_log_price_model(base_model, X_train, y_train, X_test):
    log_model = clone(base_model)
    log_model.fit(X_train, np.log1p(y_train))
    log_pred = np.expm1(log_model.predict(X_test))
    return log_model, np.maximum(log_pred, 0)


def make_log_experiment(model_name, split_strategy, fitted_raw_model, X_train, y_train, X_test, y_test):
    raw_pred = fitted_raw_model.predict(X_test)
    _, log_pred = fit_log_price_model(models[model_name], X_train, y_train, X_test)

    rows = []
    prediction_frames = []
    for target_type, pred in [('raw_price', raw_pred), ('log_price', log_pred)]:
        rows.append({
            'model': model_name,
            'split_strategy': split_strategy,
            'target_type': target_type,
            **evaluate_regression(y_test, pred),
        })
        prediction_frames.append(pd.DataFrame({
            'model': model_name,
            'split_strategy': split_strategy,
            'target_type': target_type,
            'actual_price_10k_krw': y_test.to_numpy(),
            'predicted_price_10k_krw': pred,
            'absolute_error_10k_krw': np.abs(y_test.to_numpy() - pred),
        }))

    return pd.DataFrame(rows), pd.concat(prediction_frames, ignore_index=True)

log_experiment_parts = []
log_prediction_parts = []
log_experiment_model_names = ['random_forest', 'extra_trees']

for model_name in log_experiment_model_names:
    log_time_scores, log_time_predictions = make_log_experiment(
        model_name,
        'time_2025_01_10_train_11_12_test',
        fitted_models[model_name],
        X_train,
        y_train,
        X_test,
        y_test,
    )
    log_random_scores, log_random_predictions = make_log_experiment(
        model_name,
        'random_80_20',
        random_fitted_models[model_name],
        X_train_random,
        y_train_random,
        X_test_random,
        y_test_random,
    )
    log_experiment_parts.extend([log_time_scores, log_random_scores])
    log_prediction_parts.extend([log_time_predictions, log_random_predictions])

log_experiment_scores = pd.concat(log_experiment_parts, ignore_index=True)
log_experiment_predictions = pd.concat(log_prediction_parts, ignore_index=True)
log_experiment_scores.sort_values(['split_strategy', 'model', 'target_type'])

,model,split_strategy,target_type,mae,rmse,r2
7,extra_trees,random_80_20,log_price,8254.113236,18101.202792,0.963892
6,extra_trees,random_80_20,raw_price,8390.457656,18029.294783,0.964179
3,random_forest,random_80_20,log_price,8144.461388,18384.702293,0.962753
2,random_forest,random_80_20,raw_price,8205.354502,18314.340879,0.963037
5,extra_trees,time_2025_01_10_train_11_12_test,log_price,11675.631778,20602.381239,0.949146
4,extra_trees,time_2025_01_10_train_11_12_test,raw_price,11469.523563,20059.807583,0.951790
1,random_forest,time_2025_01_10_train_11_12_test,log_price,11315.458196,21948.797750,0.942282
0,random_forest,time_2025_01_10_train_11_12_test,raw_price,11083.918145,20612.263524,0.949098


In [93]:
log_experiment_predictions['price_band'] = pd.cut(
    log_experiment_predictions['actual_price_10k_krw'],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
)

log_price_band_mae = (
    log_experiment_predictions
    .groupby(['split_strategy', 'model', 'target_type', 'price_band'], observed=True)
    .agg(
        rows=('absolute_error_10k_krw', 'size'),
        actual_mean=('actual_price_10k_krw', 'mean'),
        pred_mean=('predicted_price_10k_krw', 'mean'),
        mae=('absolute_error_10k_krw', 'mean'),
    )
    .reset_index()
    .sort_values(['split_strategy', 'model', 'target_type', 'price_band'])
)

log_price_band_mae

,split_strategy,model,target_type,price_band,rows,actual_mean,pred_mean,mae
0,random_80_20,extra_trees,log_price,<=5eok,1731,35170.814558,37498.809629,3430.115904
1,random_80_20,extra_trees,log_price,5-10eok,5739,75624.405297,77121.574496,4594.253995
2,random_80_20,extra_trees,log_price,10-15eok,4003,124045.843867,123926.591424,7072.240819
3,random_80_20,extra_trees,log_price,15-20eok,1970,172578.586802,169359.924582,10259.379717
4,random_80_20,extra_trees,log_price,20-30eok,1291,245474.449264,238142.197107,16596.565630
5,random_80_20,extra_trees,log_price,>30eok,737,429534.616011,409125.159336,34529.317910
6,random_80_20,extra_trees,raw_price,<=5eok,1731,35170.814558,38390.571628,4095.189295
7,random_80_20,extra_trees,raw_price,5-10eok,5739,75624.405297,78184.682236,5119.976568
8,random_80_20,extra_trees,raw_price,10-15eok,4003,124045.843867,125254.860157,7230.495625
9,random_80_20,extra_trees,raw_price,15-20eok,1970,172578.586802,171379.833648,10312.492463


In [94]:
fig, axes = plt.subplots(
    len(log_experiment_model_names),
    2,
    figsize=(14, 8),
    sharey=True,
)

split_strategies = log_experiment_scores['split_strategy'].unique()
for row_idx, model_name in enumerate(log_experiment_model_names):
    for col_idx, split_strategy in enumerate(split_strategies):
        ax = axes[row_idx, col_idx]
        plot_df = log_price_band_mae[
            log_price_band_mae['model'].eq(model_name)
            & log_price_band_mae['split_strategy'].eq(split_strategy)
        ]
        pivot_df = plot_df.pivot(index='price_band', columns='target_type', values='mae')
        x = np.arange(len(pivot_df.index))
        width = 0.38
        ax.bar(x - width / 2, pivot_df['raw_price'], width=width, label='raw_price', color='#4C78A8')
        ax.bar(x + width / 2, pivot_df['log_price'], width=width, label='log_price', color='#F58518')
        ax.set_title(f'{model_name} / {split_strategy}')
        ax.set_xticks(x)
        ax.set_xticklabels(pivot_df.index.astype(str), rotation=35, ha='right')
        ax.set_xlabel('Actual price band')
        ax.grid(axis='y', alpha=0.25)

axes[0, 0].set_ylabel('MAE (10k KRW)')
axes[1, 0].set_ylabel('MAE (10k KRW)')
axes[0, 0].legend()
fig.tight_layout()
plt.show()
plt.close(fig)

/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_58113/2974238742.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. 변수 중요도 확인

`RandomForestRegressor`와 `ExtraTreesRegressor`의 변수 중요도를 함께 확인한다. 범주형 변수는 순서형 인코딩 기준으로 변수 단위 중요도를 해석한다.

In [95]:
importance_model_names = ['random_forest', 'extra_trees']
importance_frames = []

for model_name in importance_model_names:
    importance_model = fitted_models[model_name].named_steps['model']
    importance_frames.append(pd.DataFrame({
        'model': model_name,
        'feature': numeric_features + categorical_features,
        'importance': importance_model.feature_importances_,
    }))

importance_df = (
    pd.concat(importance_frames, ignore_index=True)
    .sort_values(['model', 'importance'], ascending=[True, False])
)

importance_df.to_csv(PROJECT_ROOT / 'reports/model_feature_importance.csv', index=False, encoding='utf-8-sig')
importance_df

,model,feature,importance
14,extra_trees,area_m2,0.300667
27,extra_trees,nearest_business_district,0.157873
21,extra_trees,nearest_business_district_distance_km,0.141569
25,extra_trees,gu,0.108946
17,extra_trees,age,0.077864
16,extra_trees,built_year,0.076902
26,extra_trees,law_dong,0.039898
24,extra_trees,large_mart_count_within_1km,0.028485
23,extra_trees,nearest_hospital_distance_km,0.026700
22,extra_trees,nearest_subway_distance_km,0.020587


In [96]:
fig, axes = plt.subplots(1, len(importance_model_names), figsize=(14, 6), sharex=True)
if len(importance_model_names) == 1:
    axes = [axes]

for ax, model_name in zip(axes, importance_model_names):
    top_importance = (
        importance_df[importance_df['model'].eq(model_name)]
        .head(20)
        .sort_values('importance')
    )
    ax.barh(top_importance['feature'], top_importance['importance'], color='#59A14F')
    ax.set_xlabel('importance')
    ax.set_title(f'{model_name} Feature Importance')
    ax.grid(axis='x', alpha=0.25)

axes[0].set_ylabel('feature')
fig.tight_layout()
fig.savefig(IMPORTANCE_PATH, dpi=150, bbox_inches='tight')
print(f'변수 중요도 그림 저장: {IMPORTANCE_PATH}')
plt.show()
plt.close(fig)

변수 중요도 그림 저장: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/reports/figures/model_feature_importance.png


/var/folders/dn/hj27qly116330wnl76ppktb40000gn/T/ipykernel_58113/170628642.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. RandomForest와 ExtraTrees 예측 확인

테스트셋에 대해 `RandomForest`와 `ExtraTrees`가 예측한 가격을 실제 거래가와 나란히 비교한다. 가격 단위는 원본과 같은 만 원이며, 해석을 쉽게 하기 위해 억 원 단위 컬럼도 함께 만든다.

In [ ]:
prediction_model_names = ['random_forest', 'extra_trees']
prediction_info_cols = [
    'contract_date',
    'gu',
    'law_dong',
    'apartment_name',
    'area_m2',
    'floor',
    'built_year',
    'age',
]


def make_prediction_comparison(fitted, X_data, y_data, source_df, split_strategy):
    available_info_cols = [col for col in prediction_info_cols if col in source_df.columns]
    prediction_df = source_df.loc[X_data.index, available_info_cols].copy()
    prediction_df.insert(0, 'split_strategy', split_strategy)
    prediction_df['actual_price_10k_krw'] = y_data.to_numpy()
    prediction_df['actual_price_eok'] = prediction_df['actual_price_10k_krw'] / 10_000

    for model_name in prediction_model_names:
        pred = fitted[model_name].predict(X_data)
        prediction_df[f'{model_name}_pred_price_10k_krw'] = pred
        prediction_df[f'{model_name}_pred_price_eok'] = pred / 10_000
        prediction_df[f'{model_name}_abs_error_10k_krw'] = np.abs(y_data.to_numpy() - pred)
        prediction_df[f'{model_name}_abs_error_eok'] = prediction_df[f'{model_name}_abs_error_10k_krw'] / 10_000

    return prediction_df


time_prediction_comparison = make_prediction_comparison(
    fitted_models,
    X_test,
    y_test,
    modeling_df,
    'time_2025_01_10_train_11_12_test',
)

random_prediction_comparison = make_prediction_comparison(
    random_fitted_models,
    X_test_random,
    y_test_random,
    modeling_df,
    'random_80_20',
)

prediction_comparison = pd.concat(
    [time_prediction_comparison, random_prediction_comparison],
    ignore_index=True,
)

prediction_comparison.to_csv(PREDICTION_PATH, index=False, encoding='utf-8-sig')
print(f'예측 결과 저장: {PREDICTION_PATH}')
prediction_comparison.head()

In [ ]:
prediction_summary = []
for split_strategy, group in prediction_comparison.groupby('split_strategy'):
    for model_name in prediction_model_names:
        prediction_summary.append({
            'split_strategy': split_strategy,
            'model': model_name,
            'mean_abs_error_10k_krw': group[f'{model_name}_abs_error_10k_krw'].mean(),
            'median_abs_error_10k_krw': group[f'{model_name}_abs_error_10k_krw'].median(),
            'mean_abs_error_eok': group[f'{model_name}_abs_error_eok'].mean(),
            'median_abs_error_eok': group[f'{model_name}_abs_error_eok'].median(),
        })

prediction_summary_df = pd.DataFrame(prediction_summary)
prediction_summary_df.sort_values(['split_strategy', 'mean_abs_error_10k_krw'])

In [ ]:
display_cols = [
    'split_strategy',
    'contract_date',
    'gu',
    'law_dong',
    'apartment_name',
    'area_m2',
    'floor',
    'actual_price_eok',
    'random_forest_pred_price_eok',
    'random_forest_abs_error_eok',
    'extra_trees_pred_price_eok',
    'extra_trees_abs_error_eok',
]

available_display_cols = [col for col in display_cols if col in prediction_comparison.columns]
prediction_comparison[available_display_cols].sample(10, random_state=42).sort_values('split_strategy')

In [ ]:
def predict_apartment_price(feature_row, fitted, model_names=prediction_model_names):
    if isinstance(feature_row, dict):
        input_df = pd.DataFrame([feature_row])
    else:
        input_df = feature_row.copy()

    input_df = input_df[numeric_features + categorical_features]
    result = pd.DataFrame(index=input_df.index)
    for model_name in model_names:
        pred = fitted[model_name].predict(input_df)
        result[f'{model_name}_pred_price_10k_krw'] = pred
        result[f'{model_name}_pred_price_eok'] = pred / 10_000
    return result

# 아래 예시는 시간 분할 테스트셋의 첫 번째 행을 사용한다.
# 직접 예측해보고 싶으면 example_features의 값을 수정한 뒤 이 셀을 다시 실행한다.
example_features = X_test.iloc[[0]].copy()
predict_apartment_price(example_features, fitted_models)

## 13. 다음 작업

현재 노트북은 기준 모델 성능 비교까지 수행한다. 이후에는 성능이 가장 좋은 모델을 기준으로 변수 조합, 로그 타깃, 하이퍼파라미터 튜닝, 시간 분할 방식 변경을 실험한다.